# Lab 31: The Noise Decoder

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 4, *The Language Engine*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-31.ipynb)

**What you will do:** log how your comprehension of a sine-wave-speech clip jumps once you learn the words, compare how quickly that jump happens with the slower, minutes-long learning curve the book reports for a harder case, and test how an open speech-recognition model's transcription of noisy audio changes once it is given the right words as a hint.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 31 you listened to sine-wave speech — a warbling, alien-sounding signal built from just
three or four sine waves tracking the formants (the resonant frequency peaks) of a real
sentence. Before you knew the sentence, it sounded like noise. After reading the transcript or
hearing the natural version once, the same unchanged sine-wave signal became clearly
intelligible, and you could not go back to hearing it as meaningless noise.

## Record your results

Rate your own comprehension of the sine-wave clip before and after the reveal, and note how
many times you had to replay it before the words "clicked." Use a 0–10 scale for
comprehension (0 = pure noise, 10 = perfectly clear speech).

In [ ]:
# example data: replace with your own ratings for each clip you tried
trials = pd.DataFrame({
    "clip": ["clip 1", "clip 2", "clip 3"],
    "comprehension_before": [1, 0, 2],
    "comprehension_after": [9, 8, 9],
    "replays_to_click": [1, 2, 1],
})
display(trials)

trials.set_index("clip")[["comprehension_before", "comprehension_after"]].plot.bar(
    figsize=(6, 3), color=["grey", "steelblue"])
plt.ylabel("comprehension (0-10)"); plt.ylim(0, 10); plt.xticks(rotation=0); plt.show()

## Compare

Lab 31 does not give a single target number for sine-wave speech itself, but it does give a
comparison case: for the harder, more degraded signal of noise-vocoded speech, comprehension
improves "within minutes" once listeners are told the words (Davis et al., 2005) — a
gradual, multi-exposure learning curve. Sine-wave speech, by contrast, is described as
becoming clear the moment you know the sentence. Check which pattern your own data show.

In [ ]:
jump = trials.comprehension_after - trials.comprehension_before
print(f"Mean jump in comprehension: {jump.mean():.1f} points (0-10 scale)")
print(f"Mean replays needed before it clicked: {trials.replays_to_click.mean():.1f}")
if trials.replays_to_click.mean() <= 1.5:
    print("An almost immediate jump, matching the instant clarity Lab 31 describes for"
          " sine-wave speech, rather than the slower minutes-long learning curve reported"
          " for noise-vocoded speech.")
else:
    print("A more gradual improvement across replays -- closer to the noise-vocoded learning"
          " curve than to the single-hearing jump Lab 31 describes for sine-wave speech.")

## The AI side

Whisper (Radford et al., 2023) uses a language model to disambiguate noisy audio, in the
same way your knowledge of a sentence disambiguates sine-wave speech. Below, a short real
speech clip is degraded with noise until it is hard to make out, then transcribed with no
hint and with an accurate text hint supplied beforehand. Compare the two transcriptions —
accuracy should jump once the model, like you, is told what to expect.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
!pip install -q transformers datasets soundfile

import numpy as np
from datasets import load_dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained("openai/whisper-tiny.en")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny.en")

ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
audio = ds[0]["audio"]["array"]
sr = ds[0]["audio"]["sampling_rate"]
reference_text = ds[0]["text"]
print("Reference (what was actually said):", reference_text)

rng = np.random.default_rng(1)
noise = rng.normal(0, 1, len(audio))
target_snr_db = -5
scale = np.sqrt(np.mean(audio ** 2) / (np.mean(noise ** 2) * 10 ** (target_snr_db / 10)))
noisy_audio = audio + scale * noise

def transcribe(audio_array, prompt=None):
    inputs = processor(audio_array, sampling_rate=sr, return_tensors="pt")
    prompt_ids = processor.get_prompt_ids(prompt, return_tensors="pt") if prompt else None
    ids = model.generate(inputs.input_features, prompt_ids=prompt_ids)
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

hint = " ".join(reference_text.split()[:4]).lower()  # a few real opening words, as a hint
results = pd.DataFrame({
    "condition": ["no hint (noisy audio alone)", f"hint: '{hint}...'"],
    "Whisper's transcription": [transcribe(noisy_audio), transcribe(noisy_audio, prompt=hint)],
})
display(results)

If the hinted transcription is closer to the reference text, Whisper's language-model prior
has done for the noisy audio what your knowledge of the sentence did for sine-wave speech:
turned an ambiguous signal into a readable one, without changing the signal itself. The
analogy has a limit worth keeping in mind: your improvement is irreversible — you cannot
un-hear the sentence — whereas Whisper's "knowledge" is a prompt you supply fresh each time
and can remove at will; it retains nothing between calls.

## Pool the class data

Pool comprehension jumps across the class (anonymous IDs, never names), so you can see
whether the near-instant jump you may have experienced generalises, or whether some
listeners need more replays than others — a possible sign of individual differences in how
strongly a "priming" prior can act on ambiguous sound.

In [ ]:
import io
csv_text = """id,comprehension_before,comprehension_after,replays_to_click
P01,1,9,1
P02,0,8,2
P03,2,9,1
P04,1,6,3
P05,0,9,1
"""  # example data: replace with your class CSV
df = pd.read_csv(io.StringIO(csv_text))
df["jump"] = df.comprehension_after - df.comprehension_before

boot = [df.jump.sample(len(df), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean comprehension jump: {df.jump.mean():.1f} points "
      f"(95% bootstrap interval {lo:.1f}-{hi:.1f}, n = {len(df)})")
df.replays_to_click.plot.hist(bins=range(1, 6), color="steelblue", alpha=0.7, figsize=(6, 3))
plt.xlabel("replays needed before it clicked"); plt.show()

## Questions to think about

1. The lab argues that "perception is not decoding -- it is matching a prediction to data." In your own words, what would this experiment have looked like if perception really were pure decoding of the acoustic signal, with no role for prior knowledge?
2. You cannot un-hear the sine-wave sentence once you know it. Is there anything about the AI Lab's noisy-audio demonstration that is similarly irreversible, or is Whisper's "knowledge" fully disposable between runs?
3. Phonemic restoration (Warren, 1970) is a related effect — a sound replaced by noise is still "heard" as present. How is this similar to, and different from, the sine-wave speech effect you just tested?
4. Cochlear implant users initially hear speech as unintelligible buzzing and improve over weeks or months, not seconds. What does the comparison with your near-instant sine-wave-speech jump suggest about how much of speech perception is a fixed, hard-wired analysis versus a prior that itself has to be learned?

## Challenge

Try a phonemic-restoration version of this effect on your own recorded voice: record yourself
saying a short sentence, then use any audio editor to replace one consonant with a short burst
of noise or a cough. Play it for a friend without telling them which sound was replaced, and
ask them to identify it. Most listeners cannot — and often insist they heard the missing sound
clearly.

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-31.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")